# A2 Challenge Analysis: Selecting and Replaying a Command Policy

The Lab established how the released model behaves under argmax. The Challenge begins with an
application requirement. You will select an accept threshold, replay overlapping prediction
traces, and transfer the same policy into firmware.

## Notebook environment

Google Colab is the supported environment for the A2 notebooks. If this notebook is running in
Colab, the setup cell asks for the complete A2 starter ZIP, extracts it into the runtime, changes
the working directory to `Python/`, and verifies the companion files. Upload the complete ZIP when
prompted; do not upload the notebook by itself.

Local Jupyter remains usable when the notebook starts from the package's `Python/` directory.

After completing the notebook in Colab, download the executed `.ipynb` and the generated figures,
then place them in the matching locations in your local A2 package before creating the final
Canvas ZIP.

In [ ]:
import io
import os
import sys
import zipfile
from pathlib import Path, PurePosixPath

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files

    package_root = Path('/content/ECE116_A2')
    python_root = package_root / 'Python'
    if not python_root.is_dir():
        print('Upload the complete ECE116 A2 starter ZIP.')
        uploaded = files.upload()
        archive_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if len(archive_names) != 1:
            raise RuntimeError('Upload exactly one complete A2 starter ZIP.')

        archive_name = archive_names[0]
        with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
            for member_name in archive.namelist():
                member_path = PurePosixPath(member_name)
                if member_path.is_absolute() or '..' in member_path.parts:
                    raise RuntimeError(f'Unsafe archive member: {member_name}')
            archive.extractall(package_root)

    if not python_root.is_dir():
        raise RuntimeError('The uploaded ZIP does not contain Python/ at its root.')
    os.chdir(python_root)
else:
    if Path.cwd().name != 'Python':
        raise RuntimeError(
            'Start the notebook from the A2 package Python/ directory, or use Google Colab.'
        )

Path('../Documentation/Fig').mkdir(parents=True, exist_ok=True)
required_files = ['eval_lib.py', 'keyword_eval.csv', 'score_traces.csv', 'trace_manifest.csv', 'check_command_policy.py']
missing_files = [name for name in required_files if not Path(name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing companion files: {missing_files}')

print('environment:', 'Google Colab' if IN_COLAB else 'Local Jupyter')
print('working directory:', Path.cwd())
print('companion files: PASS')

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import eval_lib as ev

DATA = Path('.')
truth, scores, metadata = ev.load_evaluation(DATA / 'keyword_eval.csv')
sweep = ev.threshold_sweep(truth, scores)
for row in sweep:
    print(f"T={row['threshold']:.2f}  precision={row['precision']:.3f}  "
          f"recall={row['recall']:.3f}  false_accepts={row['false_accepts']}")

thresholds = [row['threshold'] for row in sweep]
precision = [row['precision'] for row in sweep]
recall = [row['recall'] for row in sweep]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(thresholds, precision, marker='o', label='command precision')
ax.plot(thresholds, recall, marker='s', label='command recall')
ax.set(xlabel='Accept threshold', ylabel='Metric value', ylim=(0, 1.03))
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig('../Documentation/Fig/a2_threshold_sweep.png', dpi=180)
plt.show()

## 1. Apply the supplied operating requirement

The timer policy must meet this requirement on the provided benchmark:

> Among windows accepted as `GO`, `UP`, or `STOP`, at least 97% must have the correct target label.

This is an application requirement, not a value you are expected to invent. A false accepted
command can add time, start the countdown, or stop the timer without the user's intent. The
variable `MINIMUM_PRECISION` records the requirement as `0.97`.

Use the threshold sweep to find the **lowest tested threshold** that meets the requirement. This
selection keeps as many correct commands as possible among the tested operating points while
satisfying the required precision.

In [ ]:
MINIMUM_PRECISION = 0.97  # Supplied application requirement. Do not change this value.
choice = ev.lowest_threshold_meeting(sweep, MINIMUM_PRECISION)
assert choice is not None, 'No tested threshold meets the supplied precision requirement.'
ACCEPT_THRESHOLD = None  # TODO C1a: replace None with round(choice['threshold'], 2).
print('choice:', choice)
print(f'ACCEPT_THRESHOLD: {ACCEPT_THRESHOLD:.2f}')

**TODO C1b:** Replace this cell with one short paragraph that reports the selected threshold and
its measured precision and recall from `choice`. Explain what a false accepted command would do to
the timer and why you selected the lowest qualifying threshold instead of a higher tested value.

## 2. Sequential predictions become events

The model emits overlapping predictions. A word can remain strong across several windows, but one
utterance should produce one command event. Replay the firmware rule: after firing, remain latched
for at least one complete model-window duration. Normally rearm when that time has elapsed and all
three target-command scores are below the release threshold. If scores never release, rearm after
the maximum latch time as a bounded fail-safe. A rearming window emits no event. A later high
window can be accepted after timeout, which trades possible repetition under a stuck score for
recovery from permanent lockout.

The release threshold, minimum latch time, and maximum latch time are supplied policy values. The
replay includes one class-shift edge case and one stuck-score timeout edge case. Keep all three
supplied values unchanged unless you can support a replacement with additional trace evidence.

In [ ]:
traces = ev.load_traces(DATA / 'score_traces.csv')
trace_manifest = ev.load_trace_manifest(DATA / 'trace_manifest.csv')
RELEASE_THRESHOLD = 0.55
MINIMUM_LATCH_MS = 1000
MAXIMUM_LATCH_MS = 1500
REPLAY_THRESHOLD = ACCEPT_THRESHOLD
replay = ev.replay_report(
    traces,
    trace_manifest,
    REPLAY_THRESHOLD,
    RELEASE_THRESHOLD,
    MINIMUM_LATCH_MS,
    MAXIMUM_LATCH_MS,
)
for row in replay:
    print(row)
assert all(row['pass'] for row in replay)

## 3. Inspect every policy decision

A summary can confirm the final event count, but it cannot explain a failure. The next cell prints
the state before and after every window, the strongest class, both rearm conditions, the emitted
event, and the reason for the decision. Predict the trace on paper before running the cell, then
compare each row with your prediction.

The policy contract is:

| State | Condition | New state | Event |
|---|---|---|---|
| Armed | Accepted target | Latched | Matching command |
| Armed | Otherwise | Armed | `none` |
| Latched | Normal release | Armed | `none` |
| Latched | Maximum timeout | Armed | `none` |
| Latched | Otherwise | Latched | `none` |

In [ ]:
def show_trace(trace_id, rows):
    timeline = ev.trace_policy(
        rows,
        REPLAY_THRESHOLD,
        RELEASE_THRESHOLD,
        MINIMUM_LATCH_MS,
        MAXIMUM_LATCH_MS,
    )
    print(f'\nTRACE {trace_id}')
    print('time  before   top(score)      release timeout event  after    reason')
    for row in timeline:
        print(
            f"{row['time_ms']:>4}  {row['state_before']:<8} "
            f"{row['top_label']:<7}({row['top_score']:.2f})  "
            f"{str(row['normal_release']):<7} "
            f"{str(row['maximum_timeout']):<7} "
            f"{row['event']:<6} {row['state_after']:<8} {row['reason']}"
        )
    return [row['event'] for row in timeline if row['event'] != 'none']

show_trace('class_shift_one_utterance', traces['class_shift_one_utterance'])
show_trace('maximum_latch_recovery', traces['maximum_latch_recovery'])

## 4. Design two policy tests

Create two traces using `ev.make_trace_window(time_ms, go, up, stop, unknown)`.

1. `student_normal_release` must accept one command, suppress an adjacent high window, rearm by
   normal release, emit nothing from the release window, and accept a later different command.
2. `student_maximum_latch` must remain latched immediately before `1500 ms`, rearm at `1500 ms`
   while a target score remains high, emit nothing from the timeout window, and accept a later
   command.

Choose the timestamps and scores. Record the expected event sequence before running the traces.
The supplied runner exposes every intermediate decision, so revise a test only when you can
explain why its rows did not create the condition you intended.

In [ ]:
student_normal_release = [
    # TODO C2a: add at least four windows that create normal release and a later command.
]

student_maximum_latch = [
    # TODO C2b: add windows before, at, and after the 1500 ms maximum latch.
]

student_trace_expectations = {
    'student_normal_release': [],  # TODO C2c: list the expected command events in order.
    'student_maximum_latch': [],   # TODO C2d: list the expected command events in order.
}

**TODO C2e:** Explain what each trace proves and name one incorrect policy that each trace would
expose. Refer to specific timestamps, scores, states, and expected events.

In [ ]:
student_traces = {
    'student_normal_release': student_normal_release,
    'student_maximum_latch': student_maximum_latch,
}

for trace_id, rows in student_traces.items():
    assert rows, f'{trace_id} has no windows. Complete the trace before running this cell.'
    expected = student_trace_expectations[trace_id]
    assert expected, f'{trace_id} has no expected events.'
    observed = show_trace(trace_id, rows)
    print('EXPECTED', expected)
    print('OBSERVED', observed)
    print('MATCH', observed == expected)
    assert observed == expected, f'{trace_id} did not produce its expected events.'

## 5. Transfer and implement the evaluated policy

Record `ACCEPT_THRESHOLD`, `RELEASE_THRESHOLD`, `MINIMUM_LATCH_MS`, and `MAXIMUM_LATCH_MS` in
`Arduino/Challenge_2/command_policy.cpp`. The notebook, source file, README, and serial evidence
must use the same values and the same release-or-timeout semantics.

Argmax, fixed class-to-event mapping, and model-structure syntax are supplied. Implement the
student-owned temporal decisions in the order listed in the handout. The supplied integration
sketch maps each accepted A2 command event into the unchanged A1 timer interface.

## 6. Check the C++ policy you wrote

Run `python check_command_policy.py` from the package's `Python/` folder. The self-check compiles
your policy against a lightweight model-interface stub and prints named behavior checks. Use its
trace and failure names to diagnose your implementation; do not treat the final pass count as a
substitute for explaining your own two traces.

## 7. Local-condition check

The provided benchmark estimates behavior on held-out corpus speakers. It does not guarantee
performance for your voice, room, distance, or microphone. Run at least five attempts per command
plus ten non-command speech attempts. Report the counts and conditions separately from the
benchmark.